# Legacy AnnoMI portfolio analysis

This notebook recreates the tables and figures from the original transcript-held-out experiment
using tracked result files. The current source-grouped results are reported in the README. Raw
counselling text and model checkpoints are not required.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "results").is_dir():
    raise RuntimeError("Run this notebook from the repository root.")
sys.path.insert(0, str(ROOT / "src"))
from annomi_portfolio.evidence import validate_evidence

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")

## Data split

AnnoMI-simple contains 9,699 utterances from 133 transcripts. The original classifier used
102 transcripts for training and reserved 31 entire transcripts for evaluation. This kept each
transcript in one partition, but different transcripts from the same source video could appear
in both partitions.

In [2]:
manifest = json.loads((ROOT / "data/source_manifest.json").read_text())
split = json.loads((ROOT / "results/protocol/official_split.json").read_text())
display(pd.DataFrame({
    "Dataset": [manifest["dataset"]],
    "Utterances": [manifest["rows"]],
    "Transcripts": [manifest["transcripts"]],
    "Train transcripts": [split["n_train_transcripts"]],
    "Held-out transcripts": [split["n_test_transcripts"]],
}))

,Dataset,Utterances,Transcripts,Train transcripts,Held-out transcripts
0,AnnoMI-simple,9699,133,102,31


## Model comparison

On the original 31-transcript holdout, context-aware RoBERTa improves accuracy and macro-F1
over the elastic-net baseline.

In [3]:
comparison = pd.read_csv(ROOT / "results/main/model_comparison.csv")
comparison_view = comparison[["model", "accuracy", "f1_macro", "f1_weighted", "brier_multiclass"]].copy()
comparison_view.columns = ["Model", "Accuracy", "Macro-F1", "Weighted F1", "Brier"]
display(comparison_view.style.format({
    "Accuracy": "{:.2%}", "Macro-F1": "{:.2%}", "Weighted F1": "{:.2%}", "Brier": "{:.4f}"
}).highlight_max(subset=["Accuracy", "Macro-F1", "Weighted F1"], color="#DCFCE7")
  .highlight_min(subset=["Brier"], color="#DCFCE7"))

,Model,Accuracy,Macro-F1,Weighted F1,Brier
0,Elastic-net logistic regression,77.70%,73.58%,77.44%,0.3192
1,FacebookAI/roberta-base (HP search + multi-seed),81.81%,77.44%,81.47%,0.2938


![Held-out model comparison](assets/model_comparison.png)

## Grouped uncertainty and paired correctness

Resampling is performed at transcript level. Both 95% gain intervals exclude zero, and exact
McNemar testing isolates the paired item-level difference.

In [4]:
significance = pd.read_csv(ROOT / "results/main/grouped_significance.csv")
sig_view = significance[[
    "metric", "delta_roberta_minus_baseline", "bootstrap_ci95_low", "bootstrap_ci95_high",
    "permutation_p_two_sided"
]].copy()
sig_view.columns = ["Metric", "Gain", "95% low", "95% high", "Two-sided p"]
display(sig_view.style.format({
    "Gain": "{:+.2%}", "95% low": "{:+.2%}", "95% high": "{:+.2%}", "Two-sided p": "{:.4f}"
}))

mcnemar = pd.read_csv(ROOT / "results/main/mcnemar_summary.csv")
display(mcnemar[mcnemar["Measure"].isin([
    "Held-out items", "Discordant pairs", "RoBERTa-only correct", "Baseline-only correct", "Exact p-value"
])])

,Metric,Gain,95% low,95% high,Two-sided p
0,accuracy,+4.11%,+1.77%,+6.26%,0.0080
1,f1_macro,+3.86%,+1.51%,+6.43%,0.0147


,Measure,Value,Meaning
1,Held-out items,973,Same utterances scored by both classifiers
2,Discordant pairs,130,Only these pairs drive McNemar's test
3,RoBERTa-only correct,85,RoBERTa correct while baseline is wrong
4,Baseline-only correct,45,Baseline correct while RoBERTa is wrong
7,Exact p-value,0.0006,Significant at 0.05


![Grouped confidence intervals](assets/significance_intervals.png)

## Class-level performance

Per-class metrics keep minority behaviour performance visible instead of relying only on a
single aggregate score.

In [5]:
per_class = pd.read_csv(ROOT / "results/main/per_class_metrics.csv")
f1_by_class = per_class[per_class["metric"] == "f1"].pivot(
    index="label", columns="model", values="score"
)
display(f1_by_class.style.format("{:.2%}").highlight_max(axis=1, color="#DCFCE7"))

model,Elastic-net logistic regression,FacebookAI/roberta-base (HP search + multi-seed)
label,,
other,89.49%,93.68%
question,82.02%,83.06%
reflection,68.69%,76.55%
therapist_input,54.13%,56.48%


![Per-class F1](assets/per_class_f1.png)

## Probability calibration

Temperature scaling improves RoBERTa's Brier score and expected calibration error while leaving
top-1 predictions unchanged. The sparse baseline still has the lowest ECE.

In [6]:
calibration = pd.read_csv(ROOT / "results/main/calibration.csv")
display(calibration[["model", "accuracy", "f1_macro", "brier_multiclass", "ece"]]
        .style.format({"accuracy": "{:.2%}", "f1_macro": "{:.2%}",
                       "brier_multiclass": "{:.4f}", "ece": "{:.4f}"}))

,model,accuracy,f1_macro,brier_multiclass,ece
0,Elastic-net logistic regression,77.70%,73.58%,0.3192,0.0450
1,FacebookAI/roberta-base (HP search + multi-seed) (uncalibrated),81.81%,77.44%,0.2938,0.1073
2,FacebookAI/roberta-base (HP search + multi-seed) (calibrated),81.81%,77.44%,0.2860,0.0877


## Extractive summarisation

Under the stored scoring rubric, **KMeans + MMR** has the higher overall usefulness score
(2.790 versus 2.445). BERTopic + MMR scores higher for faithfulness/support and coverage but
lower for specificity.

In [7]:
summary_methods = pd.read_csv(ROOT / "results/summarisation/method_means.csv")
rubric_columns = ["Method", "Faithfulness / Support", "Coverage", "Specificity",
                  "Non-redundancy", "Overall Usefulness"]
display(summary_methods[rubric_columns].style.format({column: "{:.3f}" for column in rubric_columns[1:]})
        .highlight_max(subset=rubric_columns[1:], color="#DCFCE7"))

,Method,Faithfulness / Support,Coverage,Specificity,Non-redundancy,Overall Usefulness
0,BERTopic + MMR,3.001,3.065,1.146,2.778,2.445
1,Pre-BERTopic KMeans + MMR,2.880,3.020,2.799,2.916,2.790


![Summarisation comparison](assets/summarisation_comparison.png)

## Original portfolio extensions

These tables show the transcript-quality and next-behaviour experiments from the original
portfolio. They use different inputs and evaluation rules from the current Tasks A and C, so the
scores are not directly comparable.

In [8]:
quality = json.loads((ROOT / "results/extensions/transcript_classification.json").read_text())
quality_view = pd.DataFrame.from_dict(quality, orient="index")[["accuracy", "f1", "roc_auc"]]
quality_view.index.name = "Model"
forecast = pd.read_csv(ROOT / "results/extensions/next_behaviour_forecasting_topk.csv")
display(quality_view.style.format("{:.3f}"))
display(forecast.style.format({"top_1_accuracy": "{:.2%}", "top_2_accuracy": "{:.2%}",
                               "top_3_accuracy": "{:.2%}"}))

,accuracy,f1,roc_auc
Model,,,
Elastic-net logistic regression,0.903,0.727,0.954
XGBoost,0.903,0.727,0.946


,model,top_1_accuracy,top_2_accuracy,top_3_accuracy
0,Majority-class baseline (always predicts 'other'),32.58%,45.12%,71.53%
1,CatBoost structured state model,44.60%,76.16%,92.19%
2,Hybrid GRU with local-state fusion,44.60%,72.56%,90.75%


## Validation

The final cell checks the stored metrics, intervals, calibration results, split, and dataset
provenance.

In [9]:
checks = validate_evidence(ROOT)
for check in checks:
    print(f"PASS  {check}")

PASS  legacy aggregate metrics match the recorded results
PASS  reported deltas reproduce direct arithmetic
PASS  transcript-grouped intervals and permutation tests support the gain
PASS  paired correctness counts are internally consistent
PASS  calibration improvement matches the recorded temperature fit
PASS  summarisation results match the aggregate ranking
PASS  project-defined transcript split is disjoint and exhaustive
PASS  dataset commit and checksum match the recorded values


## Scope

These results apply only to the original transcript holdout. They do not establish clinical
validity, demographic fairness, or safe use in patient-facing systems. See
`docs/MODEL_CARD.md` for the full limitations statement.